# PDF Processing via LLMaaS

## 0. Setting your Name and Email

Please starting by putting your name and email in the following variables - please stick to the required format i.e. NAME_SURNAME

In [1]:
# WRITE YOUR NAME_SURNAME HERE, AS WELL AS YOUR EMAIL WITH WHICH YOU LOGGED IN INTO CELONIS
MY_NAME = 'NIEL_GEORGE_VARGHESE'
MY_EMAIL = 'niel.x.varghese@gmail.com'

## 1. Extract information from Invoices

This is the section you will need to fill in. Your code should create the following
- **a pandas dataframe called df that includes the extracted information**.
- **the dataframe should contain a column called 'po_reference' that contains the reference to the PO**
- **the values in the column 'po_reference' should be a 11-char long strings. Use left padding with zeros where needed.**

In [17]:
# One-time setup (uncomment if running locally / in Colab):
# %pip install anthropic pymupdf pandas
#
# Auth: you will be prompted for your Anthropic API key if it isn't already set
# (so the key never ends up in the code you submit).

import os, re, json, glob, base64, getpass
import pandas as pd
import requests
import fitz  # PyMuPDF - renders PDF pages to images, no poppler needed
import anthropic

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

INVOICE_DIR = "./Invoices"
IMAGE_OUT_DIR = "./invoice_images"   # images are saved here, then uploaded to GitHub

# --- GitHub image hosting (one-time setup: create an empty PUBLIC repo on your own account,
#     tick "Add a README", and create a fine-grained token with Contents: Read and write on it) ---
GITHUB_USER = "<your-github-username>"
GITHUB_REPO = "<your-public-repo-name>"
GITHUB_BRANCH = "main"
GITHUB_FOLDER = "invoice_images"
if not os.environ.get("GITHUB_TOKEN"):
    os.environ["GITHUB_TOKEN"] = getpass.getpass("GitHub token (not saved in the code): ")
IMAGE_BASE_URL = f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}/{GITHUB_FOLDER}"

MODEL = "claude-sonnet-5-5"   # any vision-capable Claude model works
FIELDS = ["vendor_name", "vendor_address", "payment_terms",
          "invoice_value", "company_code", "po_reference", "invoice_id"]

PROMPT = """You are extracting data from an invoice. Return ONLY a JSON object (no prose,
no markdown fences) with exactly these keys:
- vendor_name: name of the company issuing the invoice
- vendor_address: the vendor's full address on one line
- payment_terms: payment terms exactly as printed (e.g. "Net 30 Days")
- invoice_value: the total amount due as a plain number with a dot as decimal separator
  (no currency symbol, no thousands separator)
- company_code: the buyer's company code (e.g. "CompanyCode3")
- po_reference: the purchase order number, digits as printed
- invoice_id: the invoice number
If a field is not present, use null. Do not guess."""

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment

MEDIA_TYPES = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg"}


def load_pages(path):
    """Return a list of (media_type, raw_bytes), one entry per page/image."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        with fitz.open(path) as doc:
            return [("image/png", page.get_pixmap(dpi=200).tobytes("png")) for page in doc]
    if ext in MEDIA_TYPES:
        with open(path, "rb") as f:
            return [(MEDIA_TYPES[ext], f.read())]
    raise ValueError(f"Unsupported file type: {path}")


def save_invoice_image(path, invoice_id):
    """Save page 1 of the invoice as an image named after invoice_id; return the local file path."""
    os.makedirs(IMAGE_OUT_DIR, exist_ok=True)
    media_type, raw = load_pages(path)[0]
    ext = "jpg" if media_type == "image/jpeg" else "png"
    fname = re.sub(r"[^\w\-]", "_", str(invoice_id)) + "." + ext
    local = os.path.join(IMAGE_OUT_DIR, fname)
    with open(local, "wb") as f:
        f.write(raw)
    return local


def upload_to_github(local_path):
    """Create or update the image in the GitHub repo and return its raw.githubusercontent URL."""
    fname = os.path.basename(local_path)
    api = f"https://api.github.com/repos/{GITHUB_USER}/{GITHUB_REPO}/contents/{GITHUB_FOLDER}/{fname}"
    headers = {"Authorization": f"Bearer {os.environ['GITHUB_TOKEN']}",
               "Accept": "application/vnd.github+json"}
    existing = requests.get(api, headers=headers, params={"ref": GITHUB_BRANCH})
    body = {"message": f"Add/update {fname}",
            "branch": GITHUB_BRANCH,
            "content": base64.b64encode(open(local_path, "rb").read()).decode()}
    if existing.status_code == 200:          # file already there -> update it (needs its sha)
        body["sha"] = existing.json()["sha"]
    r = requests.put(api, headers=headers, json=body)
    if r.status_code not in (200, 201):
        raise RuntimeError(f"GitHub upload failed [{r.status_code}]: {r.text[:200]}")
    return f"{IMAGE_BASE_URL}/{fname}"


def extract_invoice(path):
    """Send the invoice image(s) to a vision LLM and return the parsed fields as a dict."""
    content = [
        {"type": "image",
         "source": {"type": "base64", "media_type": mt,
                    "data": base64.b64encode(raw).decode()}}
        for mt, raw in load_pages(path)
    ]
    content.append({"type": "text", "text": PROMPT})
    resp = client.messages.create(
        model=MODEL, max_tokens=1000,
        messages=[{"role": "user", "content": content}],
    )
    text = "".join(b.text for b in resp.content if b.type == "text")
    match = re.search(r"\{.*\}", text, re.DOTALL)   # tolerate stray text around the JSON
    if not match:
        raise ValueError(f"No JSON found in model output: {text[:200]}")
    return json.loads(match.group(0))


def clean(rec):
    """Normalise types/formats so the Celonis join on po_reference works."""
    out = {k: rec.get(k) for k in FIELDS}
    # PO reference -> digits only, left-padded with zeros to 11 chars
    out["po_reference"] = re.sub(r"\D", "", str(out["po_reference"] or "")).zfill(11)
    # Invoice value -> float
    val = re.sub(r"[^\d.\-]", "", str(out["invoice_value"] or ""))
    out["invoice_value"] = float(val) if val else None
    for k in ("vendor_name", "vendor_address", "payment_terms", "company_code", "invoice_id"):
        if isinstance(out[k], str):
            out[k] = out[k].strip()
    return out


files = sorted(f for f in glob.glob(os.path.join(INVOICE_DIR, "*"))
               if os.path.splitext(f)[1].lower() in {*MEDIA_TYPES, ".pdf"})
print(f"Found {len(files)} invoice file(s) in {INVOICE_DIR}")

rows, failed = [], []
for f in files:
    try:
        rec = clean(extract_invoice(f))
        rec["invoice_image_url"] = upload_to_github(save_invoice_image(f, rec["invoice_id"]))   # bonus: link to original invoice
        rows.append(rec)
        print(f"  OK     {os.path.basename(f)}")
    except Exception as e:
        failed.append(f)
        print(f"  FAILED {os.path.basename(f)}: {e}")

if not rows:
    raise RuntimeError("No invoices could be extracted - check API key and the Invoices folder.")
if failed:
    print(f"WARNING: {len(failed)} file(s) were skipped and will NOT be pushed.")

df = pd.DataFrame(rows, columns=FIELDS + ["invoice_image_url"])
assert df["po_reference"].str.len().eq(11).all(), "po_reference must be 11 chars"
df

Found 5 invoice file(s) in ./Invoices
  OK      INV-2020-08-001247.pdf
  OK     INV-2020-001.png
  OK     INV-2020-07-001853.pdf
  OK     INV-2021-001.png
  OK     INV-2021-002.png
NOTE: set IMAGE_BASE_URL to your real public image host before pushing.


,vendor_name,vendor_address,payment_terms,invoice_value,company_code,po_reference,invoice_id,invoice_image_url
0,Vendor40 Corporation,"5678 Vendor Street, Business District, Chicago...",Net 30 Days,3960.0,CompanyCode3,00000586652,INV-2020-08-001247,https://raw.githubusercontent.com/<github-user...
1,Vendor34,"789 Vendor Street, Vendor City, State 54321",Payment is due within 30 days of invoice date....,1666.0,CompanyCode4,00000048334,INV-2020-001,https://raw.githubusercontent.com/<github-user...
2,Vendor46 Manufacturing Solutions,"Manufacturing District 12, Hamburg, HH 20095, ...",NET 90 DAYS,3748.5,CompanyCode3,00001135572,INV-2020-07-001853,https://raw.githubusercontent.com/<github-user...
3,Vendor45,"789 Vendor Street, Vendor City, State 54321",Payment is due within 30 days of invoice date....,5057.5,CompanyCode3,00000000181,INV-2021-001,https://raw.githubusercontent.com/<github-user...
4,Vendor34,"789 Vendor Street, Vendor City, State 54321",Payment is due within 15 days of invoice date....,1785.0,CompanyCode4,00000000191,INV-2021-002,https://raw.githubusercontent.com/<github-user...


In [11]:
   print(df.columns.tolist())
   df["invoice_image"].str.len()

['vendor_name', 'vendor_address', 'payment_terms', 'invoice_value', 'company_code', 'po_reference', 'invoice_id', 'invoice_image']


0     69867
1     93055
2    108743
3     94655
4     56403
Name: invoice_image, dtype: int64

## 2. Pushing Data back to Celonis

The below script will push your results into Celonis, it may take **up to 5 minutes** for the results to show up in Celonis frontend. Please refrain from triggering the below script multiple times in a row.

In [16]:
%run push.ipynb

Successfully pushed file to Celonis
